<a href="https://colab.research.google.com/github/chaiyawat19/DataScinceLab/blob/main/DS4marketingExample_RecommendationSystemMovies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np # linear algebra
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
ratings=pd.read_csv('/content/ratings.csv')
ratings.head(20)

In [ ]:
movies=pd.read_csv('/content/movies.csv')
movies.head(15)

In [ ]:
ratings.shape

In [ ]:
movies.shape

Merge Movies and rating table

In [ ]:
movie_ratings = pd.merge(movies, ratings)
movie_ratings

In [ ]:
ratings_matrix = ratings.pivot_table(index=['userId'],columns=['movieId'],values='rating').reset_index(drop=True)
ratings_matrix.fillna( 0, inplace = True )
ratings_matrix

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
ratings=pd.read_csv('/content/ratings.csv')
movies=pd.read_csv('/content/movies.csv')
movie_ratings = pd.merge(movies, ratings)
ratings_matrix = ratings.pivot_table(index=['movieId'],columns=['userId'],values='rating').reset_index(drop=True)
ratings_matrix.fillna( 0, inplace = True )

print(f"Unique User IDs: {ratings['userId'].nunique()}")
print(f"Shape of ratings matrix: {ratings_matrix.shape}")


In [ ]:
movie_similarity=cosine_similarity(ratings_matrix)
np.fill_diagonal(movie_similarity, 0 )
ratings_matrix = pd.DataFrame( movie_similarity )
ratings_matrix.head(15)

Item-Based Filtering

In [ ]:
try:
    movie_inp=input('Enter the reference movie title based on which recommendations are to be made: ')
    inp=movies[movies['title']==movie_inp].index.tolist()
    inp=inp[0]

    movies['similarity'] = ratings_matrix.iloc[inp]
    movies.head(5)

except:
    print("Sorry, the movie is not in the database!")

print("Recommended movies ",movie_inp ,"\n", movies.sort_values( ["similarity"], ascending = False )[0:10])

User-Based Filtering

In [ ]:
def get_recommendations(user_id, top_n=10):
    try:
        user_ratings = ratings[ratings['userId'] == user_id]
        if user_ratings.empty:
          return "User not found in the dataset."
        user_movie_ids = user_ratings['movieId'].unique()
        # Calculate average similarity for movies the user has rated
        similarities = []
        for movie_id in user_movie_ids:
            movie_index = movies[movies['movieId'] == movie_id].index[0]  # Get the index
            similarities.append(ratings_matrix.iloc[movie_index])

        average_similarity = pd.concat(similarities, axis=1).mean(axis=1)

        # Sort movies based on average similarity and recommend top_n movies
        movies['similarity'] = average_similarity
        recommendations = movies.sort_values(by=['similarity'], ascending=False)
        return recommendations.head(top_n)

    except Exception as e:
        return f"An error occurred: {e}"

# Get recommendations for a specific user
user_id_input = int(input("Enter the user ID: "))
recommendations = get_recommendations(user_id_input)
recommendations